# 15-Minute Counterfactual Baseline — XGBoost vs Temporal Fusion Transformer (TFT)

## Pedagogical and reproducible notebook (Google Colab / Jupyter)

This notebook explains **step by step** how to build a **counterfactual baseline** of electricity consumption at **15-minute** granularity, then compares two model families: **XGBoost** and the **Temporal Fusion Transformer (TFT)**.

> 🔒 **Anonymized version**: all data is **simulated**. No real source, no company name, no internal table. The notebook is **runnable end-to-end** without a cluster or credentials.

### What is a counterfactual baseline?

During a **demand-management event** (load curtailment, capacity call), an electricity provider asks customers to reduce their consumption. To measure the real effect of the event, we must answer a **counterfactual** question:

> *"What **would** consumption have been if there had been **no** event?"*

This hypothetical consumption is the **baseline**. By comparing the predicted baseline to the actual consumption on event days, we measure:

| Quantity | Definition |
|----------|-----------|
| **Curtailed energy** | Deficit on event day D (baseline − actual) |
| **Pre-charge** | Surplus the day before D-1 (anticipation) |
| **Rebound** | Surplus the day after D+1 (deferred load) |

**Net avoided energy** = curtailed energy − pre-charge − rebound.

### What you will learn
1. Simulate realistic 15-min consumption data (with weather, events, maintenance).
2. Automatically detect maintenance days.
3. Engineer features (calendar, cyclical encoding, lagged weather, reference profiles).
4. Train a robust **XGBoost** baseline (residual target, monotonicity, debiasing).
5. Train a **TFT** baseline without information leakage.
6. Compare both models and measure load shifting.

## 0. Setting up the environment

We import the scientific libraries. `XGBoost` and `scikit-learn` are enough for the main part; `torch`, `lightning` and `pytorch-forecasting` are only needed for Part 2 (TFT) and will be installed at that point.

In [ ]:
# In Colab, uncomment if needed:
# %pip install -q xgboost scikit-learn numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (16, 5)
plt.rcParams["axes.grid"] = True

SEED = 42
np.random.seed(SEED)
print("Environment ready. NumPy", np.__version__, "| pandas", pd.__version__)

## 1. Global parameters

All configuration is centralized here. The granularity is **15 minutes** (96 slots per day, no hourly aggregation). The **backtest** is performed around an event date `DATE_TEST`, with a ±8-day window.

The **flags** enable/disable model improvements: residual target, physical monotonicity, local debiasing, clipping and maintenance detection. This lets us study the effect of each technique in isolation.

In [ ]:
# ---- 15-minute granularity ----
FREQ          = "15min"
STEPS_PER_HOUR = 4
STEPS_PER_DAY  = 24 * STEPS_PER_HOUR      # 96 slots/day

# ---- Backtest window around an event ----
DATE_TEST      = "2026-03-03"
N_DAYS_BEFORE  = 8
N_DAYS_AFTER   = 8

# ---- Event phases (in 15-min slots) ----
PRE_STEPS  = 2 * STEPS_PER_HOUR            # 2 h before
POST_STEPS = 2 * STEPS_PER_HOUR            # 2 h after

# ---- Baseline-train exclusions ----
EXCLUDE_EVENT_DAY   = True
EXCLUDE_DAY_BEFORE  = True
EXCLUDE_DAY_AFTER   = True
EXCLUDE_MAINTENANCE = True

# ---- Toggleable improvements ----
RESIDUAL_TARGET     = True             # target = energy - reference profile
USE_MONOTONE        = True             # increasing constraints on HDD/CDD
XGB_OBJECTIVE       = "reg:pseudohubererror"   # robust to outliers
LOCAL_DEBIASING     = True             # bias recalibration on a recent window
DEBIAS_CALIB_DAYS   = 30
CLIP_PREDICTIONS    = True
CLIP_Q_LOW, CLIP_Q_HIGH = 0.001, 0.999
DETECT_MAINTENANCE  = True

# ---- Maintenance detection ----
MAINT_RATIO_THR = 0.75
MAINT_CV_THR    = 0.12
MAINT_N_NEIGHBORS = 4

DATE_TEST_TS = pd.Timestamp(DATE_TEST).normalize()
test_start = DATE_TEST_TS - pd.Timedelta(days=N_DAYS_BEFORE)
test_end   = DATE_TEST_TS + pd.Timedelta(days=N_DAYS_AFTER + 1) - pd.Timedelta(minutes=15)
print(f"Granularity: {FREQ} ({STEPS_PER_DAY} steps/day)")
print("Test date:", DATE_TEST_TS.date(), "| Window:", test_start, "->", test_end)

## 2. Simulating realistic consumption data

We simulate **4 anonymized customer segments**, each with its own scale and behavior:

| Segment | Profile | Energy scale |
|---------|---------|--------------|
| **Residential** | Morning/evening peaks, cold-sensitive | ~0.4 – 1.1 |
| **Business** | High on weekdays, low on weekends | ~5 – 8 |
| **Controlled** | Residential + strong curtailments | ~0.4 – 1.7 |
| **Behavioral** | Soft, non-controlled response | ~0.4 – 1.3 |

### Simulation ingredients
Each series is built as a **sum of known physical components** (the *ground truth*):
- a daily and weekly **base** (living/working habits);
- a **weather** effect: the colder it is (high HDD), the higher the consumption;
- **curtailment events** that reduce consumption for a few hours, with **pre-charge** before and **rebound** after;
- **maintenance days** with abnormally low consumption;
- random **noise**.

Since we know the ground truth, we can **verify** that the model recovers the correct behavior.

In [ ]:
# ---- Time horizon (simulated): ~5 months at 15 min ----
START = pd.Timestamp("2025-11-15")
END   = pd.Timestamp("2026-03-20")
timestamps = pd.date_range(START, END, freq=FREQ, inclusive="left")
N = len(timestamps)
print("Number of 15-min slots:", N, "| from", timestamps[0], "to", timestamps[-1])

# ---- Simulated weather: temperature -> HDD (cold) and CDD (hot) ----
day_of_year = timestamps.dayofyear.values
hour_frac = timestamps.hour.values + timestamps.minute.values/60.0
# Temperature: seasonal cycle (cold winter) + daily cycle + noise
temp_season = 2.0 - 12.0*np.cos(2*np.pi*(day_of_year-15)/365.0)   # min ~ mid-January
temp_daily  = 3.0*np.sin(2*np.pi*(hour_frac-9)/24.0)              # warmer in the afternoon
temperature = temp_season + temp_daily + np.random.normal(0, 1.5, N)
HDD = np.clip(15.0 - temperature, 0, None)   # heating degree units (cold)
CDD = np.clip(temperature - 18.0, 0, None)   # cooling degree units (hot)

weather = pd.DataFrame({
    "timestamp": timestamps,
    "temperature": temperature.round(2),
    "hdd": HDD.round(3),
    "cdd": CDD.round(3),
    "humidex": (temperature + np.random.normal(0, 1, N)).round(2),
    "wind":    np.clip(np.random.gamma(2.0, 6.0, N), 0, None).round(1),
})
weather.head()

### 2.1 Event and maintenance dates

We set **event dates** (curtailment) around the test period, and a few **maintenance days**. These lists are the ground truth: the model will not know them, but we will use them for evaluation.

In [ ]:
# Event (curtailment) days: a few spread-out days, including DATE_TEST
event_days = pd.to_datetime([
    "2026-02-24", "2026-02-25", "2026-02-27",
    "2026-03-02", "2026-03-03", "2026-03-06",
])
# Curtailment slots within the day (e.g. morning peak 6-9am and evening peak 5-8pm)
CURTAILMENT_SLOTS = [(6, 9), (17, 20)]

# Maintenance days (abnormally low consumption, no event)
maintenance_days = pd.to_datetime(["2026-01-19", "2026-02-09"])

print("Events     :", [d.date().isoformat() for d in event_days])
print("Maintenance:", [d.date().isoformat() for d in maintenance_days])

### 2.2 Segment generation function

`generate_segment(...)` combines all components to produce a 15-min series. We deliberately separate:
- `energy` = the **observed** consumption (what the model sees);
- `event`  = the curtailment indicator (1 during curtailment slots on event days).

An event removes a fraction of load during curtailment, adds a **pre-charge** the day before and a **rebound** the day after — exactly the behavior a good baseline must learn to ignore.

In [ ]:
def daily_profile(hour_frac, peaks, width=2.2):
    """Sum of Gaussians = consumption peaks throughout the day."""
    y = np.zeros_like(hour_frac)
    for h, amp in peaks:
        y += amp * np.exp(-0.5*((hour_frac - h)/width)**2)
    return y

def generate_segment(name, base, weather_amp, peaks, weekly_weight, curtailment_strength,
                     noise=0.03, scale=1.0):
    hf = timestamps.hour.values + timestamps.minute.values/60.0
    dow = timestamps.dayofweek.values                      # 0=Mon ... 6=Sun
    is_weekend = (dow >= 5).astype(float)

    # Base: daily profile + weekly modulation
    load = base + daily_profile(hf, peaks)
    load *= (1.0 - weekly_weight*is_weekend)               # lower on weekends

    # Weather effect: cold increases consumption
    load += weather_amp * HDD + 0.15*weather_amp * CDD

    # Light multiplicative noise
    load *= (1.0 + np.random.normal(0, noise, N))

    d = pd.DataFrame({"timestamp": timestamps, "energy": load})
    d["date"] = d["timestamp"].dt.normalize()
    d["hour"] = d["timestamp"].dt.hour

    # Event (curtailment) indicator
    d["event"] = 0
    evt_dates = set(event_days.normalize())
    for (h0, h1) in CURTAILMENT_SLOTS:
        mask = d["date"].isin(evt_dates) & d["hour"].between(h0, h1-1)
        d.loc[mask, "event"] = 1

    # Curtailment: remove a fraction during the event
    d.loc[d["event"] == 1, "energy"] *= (1.0 - curtailment_strength)
    # Pre-charge the day before (slight evening surplus) and rebound the day after
    for j in event_days.normalize():
        day_before = j - pd.Timedelta(days=1)
        day_after  = j + pd.Timedelta(days=1)
        m_pre = (d["date"] == day_before) & d["hour"].between(20, 22)
        m_reb = (d["date"] == day_after)  & d["hour"].between(6, 9)
        d.loc[m_pre, "energy"] *= (1.0 + 0.5*curtailment_strength)
        d.loc[m_reb, "energy"] *= (1.0 + 0.4*curtailment_strength)

    # Maintenance days: abnormally low consumption
    for j in maintenance_days.normalize():
        d.loc[d["date"] == j, "energy"] *= 0.55

    d["energy"] = (scale * d["energy"]).clip(lower=0.01)
    return d[["timestamp", "energy", "event"]]

# Configuration of the 4 segments (scales inspired by real profiles)
SEGMENT_CONFIG = {
    "Residential":  dict(base=0.45, weather_amp=0.012, peaks=[(7,0.35),(19,0.5)],
                         weekly_weight=0.10, curtailment_strength=0.18, scale=1.0),
    "Business":     dict(base=5.2,  weather_amp=0.05,  peaks=[(11,1.6),(15,1.5)],
                         weekly_weight=0.45, curtailment_strength=0.10, scale=1.0),
    "Controlled":   dict(base=0.5,  weather_amp=0.015, peaks=[(7,0.45),(19,0.6)],
                         weekly_weight=0.12, curtailment_strength=0.35, scale=1.0),
    "Behavioral":   dict(base=0.5,  weather_amp=0.013, peaks=[(7,0.4),(19,0.55)],
                         weekly_weight=0.11, curtailment_strength=0.15, scale=1.0),
}

segment_data = {name: generate_segment(name, **cfg) for name, cfg in SEGMENT_CONFIG.items()}
for name, d in segment_data.items():
    print(f"{name:14s}: {len(d)} points | mean energy = {d['energy'].mean():.3f} "
          f"| min={d['energy'].min():.2f} max={d['energy'].max():.2f}")

### 2.3 Visual preview of a segment

Let's plot one week of the **Controlled** segment to see the curtailments (red bands): consumption drops during event slots.

In [ ]:
d = segment_data["Controlled"]
week = d[(d["timestamp"] >= "2026-03-01") & (d["timestamp"] < "2026-03-08")]
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(week["timestamp"], week["energy"], color="black", lw=1.1, label="Observed energy")
for j in week.loc[week["event"] == 1, "timestamp"]:
    ax.axvspan(j, j + pd.Timedelta(minutes=15), color="red", alpha=0.15, lw=0)
ax.set_title("Controlled segment — one week (red bands = curtailment slots)")
ax.set_ylabel("Energy (15 min)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
ax.legend(); plt.tight_layout(); plt.show()

## 3. Automatic maintenance-day detection

A baseline must learn **normal** behavior. **Maintenance days** (abnormally low consumption for technical reasons) would distort learning: we need to spot and **exclude** them.

### Intuitive idea
For each day, we compare it to its **same-weekday neighbors** (a Monday is compared to other Mondays). If its consumption is **abnormally low** (`ratio ≤ 0.75`) **and** has **low variability** (low coefficient of variation) **outside of events**, we flag it as maintenance.

In [ ]:
def detect_maintenance(d):
    d = d.copy()
    d["date"] = d["timestamp"].dt.normalize()
    # Daily aggregation
    day = d.groupby("date").agg(
        day_energy=("energy", "sum"),
        mean=("energy", "mean"),
        std=("energy", "std"),
        n_event=("event", "sum"),
    ).reset_index()
    day["dow"] = day["date"].dt.dayofweek
    day["cv"] = day["std"] / day["mean"].replace(0, np.nan)

    # Local reference: median of same-weekday neighbors
    day = day.sort_values("date").reset_index(drop=True)
    ref = np.full(len(day), np.nan)
    for dow in range(7):
        idx = day.index[day["dow"] == dow].tolist()
        vals = day.loc[idx, "day_energy"].values
        for k, gi in enumerate(idx):
            lo, hi = max(0, k-MAINT_N_NEIGHBORS), min(len(idx), k+MAINT_N_NEIGHBORS+1)
            neighbors = np.concatenate([vals[lo:k], vals[k+1:hi]])
            if len(neighbors):
                ref[gi] = np.median(neighbors)
    day["ref_local"] = ref
    day["ratio"] = day["day_energy"] / day["ref_local"]

    day["flag_maintenance"] = (
        (day["ratio"] <= MAINT_RATIO_THR) &
        (day["cv"] <= MAINT_CV_THR) &
        (day["n_event"] == 0)
    ).astype(int)
    return day[["date", "ratio", "cv", "flag_maintenance"]]

# Test on the Residential segment
flags = detect_maintenance(segment_data["Residential"])
detected = flags.loc[flags["flag_maintenance"] == 1, "date"].dt.date.tolist()
print("Detected maintenance days:", detected)
print("Ground truth             :", [j.date() for j in maintenance_days])

## 4. Feature engineering

The model needs explanatory variables. We create three families.

### 4.1 Calendar and cyclical features
Hour and day are **cyclical**: 11pm is close to midnight. A raw encoding would create a false discontinuity. We therefore use `sin`/`cos` of the intra-day slot (`slot`, 0→95) and of the day of week.

### 4.2 Lagged weather (24h lag / delta)
Buildings have **thermal inertia**: yesterday's cold still influences today. We add the weather **shifted by 24h** and its 24h **change**.

### 4.3 Event features
We mark event slots, the **pre/post** phases, and the **D-1 / D+1** days. They are used for train exclusion and load-shifting analysis (never as baseline features).

In [ ]:
def add_event_features(d):
    d = d.copy().sort_values("timestamp").reset_index(drop=True)
    d["date"] = d["timestamp"].dt.normalize()
    d["hour"] = d["timestamp"].dt.hour
    d["event"] = d["event"].fillna(0).astype(int)
    d["is_event_day"] = d.groupby("date")["event"].transform("max").astype(int)

    # Pre/post curtailment phases
    d["phase_pre"], d["phase_post"] = 0, 0
    idx = np.where(d["event"].values == 1)[0]; n = len(d)
    for i in idx:
        for k in range(1, PRE_STEPS+1):
            if i-k >= 0 and d["event"].iat[i-k] == 0:
                d.iat[i-k, d.columns.get_loc("phase_pre")] = 1
        for k in range(1, POST_STEPS+1):
            if i+k < n and d["event"].iat[i+k] == 0:
                d.iat[i+k, d.columns.get_loc("phase_post")] = 1

    # Neighboring days D-1 / D+1
    evt_days = set(d.loc[d["is_event_day"] == 1, "date"].unique())
    d["day_before_event"], d["day_after_event"] = 0, 0
    for j in evt_days:
        d.loc[d["date"] == j - pd.Timedelta(days=1), "day_before_event"] = 1
        d.loc[d["date"] == j + pd.Timedelta(days=1), "day_after_event"] = 1
    d.loc[d["is_event_day"] == 1, ["day_before_event", "day_after_event"]] = 0
    return d

def add_features(d, weather):
    d = d.merge(weather, on="timestamp", how="left").sort_values("timestamp").reset_index(drop=True)
    d["hour"]  = d["timestamp"].dt.hour
    d["minute"] = d["timestamp"].dt.minute
    d["dow"]   = d["timestamp"].dt.dayofweek
    d["month"] = d["timestamp"].dt.month
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["slot"]  = d["hour"]*STEPS_PER_HOUR + d["minute"]//15
    # Cyclical encodings
    d["slot_sin"] = np.sin(2*np.pi*d["slot"]/STEPS_PER_DAY)
    d["slot_cos"] = np.cos(2*np.pi*d["slot"]/STEPS_PER_DAY)
    d["dow_sin"]  = np.sin(2*np.pi*d["dow"]/7)
    d["dow_cos"]  = np.cos(2*np.pi*d["dow"]/7)
    # One-hot of day of week
    for j in range(7):
        d[f"dow_{j}"] = (d["dow"] == j).astype(int)
    # Temperature interactions
    d["hdd_x_slot"]    = d["hdd"] * d["slot"]
    d["hdd_x_weekend"] = d["hdd"] * d["is_weekend"]
    # 24h lagged weather + delta
    for col in ["hdd", "cdd", "humidex", "wind"]:
        d[f"{col}_lag24"]   = d[col].shift(STEPS_PER_DAY)
        d[f"delta_{col}24"] = d[col] - d[col].shift(STEPS_PER_DAY)
    return d

### 4.4 Per-slot reference profiles (the anti-NaN key)

In **residual-target** mode, the model predicts `energy − reference_profile`. The **reference profile** is the average consumption of a "normal" slot (e.g. Monday at 8:00am).

⚠️ **Classic pitfall**: if a slot never has any "clean" data, its profile would be `NaN`, and since the target is `energy − profile`, a single `NaN` makes training fail. We therefore build the profile with a **progressive fallback**: (day of week, slot) → (month, day, slot) → (slot only) → **global mean**. Result: **never any NaN**.

In [ ]:
def build_baseline_mask(d):
    """Select 'clean' rows to learn the baseline."""
    mask = pd.Series(True, index=d.index)
    if EXCLUDE_EVENT_DAY:  mask &= d["is_event_day"].eq(0)
    if EXCLUDE_DAY_BEFORE: mask &= d["day_before_event"].eq(0)
    if EXCLUDE_DAY_AFTER:  mask &= d["day_after_event"].eq(0)
    if EXCLUDE_MAINTENANCE and "flag_maintenance" in d.columns:
        mask &= d["flag_maintenance"].eq(0)
    return mask

def add_reference_profile(d, baseline_mask):
    d = d.copy().reset_index(drop=True)
    d["week_of_year"] = d["timestamp"].dt.isocalendar().week.astype(int)
    base = d.loc[baseline_mask]
    def prof(keys, name):
        g = base.groupby(keys)["energy"].mean().rename(name).reset_index()
        return d.merge(g, on=keys, how="left")[name]
    p1 = prof(["dow", "slot"], "p1")
    p2 = prof(["month", "dow", "slot"], "p2")
    p3 = prof(["slot"], "p3")
    global_mean = float(base["energy"].mean())
    d["ref_profile"] = (p2.fillna(p1).fillna(p3).fillna(global_mean))
    n_nan = int(d["ref_profile"].isna().sum())
    if n_nan:
        d["ref_profile"] = d["ref_profile"].fillna(global_mean)
    return d

FEATURES = [
    "slot", "dow", "month", "is_weekend",
    "slot_sin", "slot_cos", "dow_sin", "dow_cos",
    *[f"dow_{j}" for j in range(7)],
    "hdd", "cdd", "humidex", "wind", "temperature",
    "hdd_x_slot", "hdd_x_weekend",
    "hdd_lag24", "cdd_lag24", "delta_hdd24", "delta_cdd24",
    "humidex_lag24", "wind_lag24",
]
print("Number of features:", len(FEATURES))

## 5. Robust XGBoost model

### 5.1 Why these choices?
- **Residual target** (`energy − ref_profile`): the model only learns the **deviation** from the usual profile, which is simpler and more accurate.
- **Monotonic constraints** on HDD/CDD: we **impose** that the baseline **increases** as cold increases — physical coherence guaranteed.
- **Pseudo-Huber objective**: robust to outliers (less sensitive than squared error).
- **Local debiasing**: we recalibrate any residual bias over the 30 days before the test.
- **Clipping**: we bound predictions between two quantiles to avoid absurd values.

In [ ]:
def build_monotone(feats):
    """Monotonicity vector: +1 (increasing) for cold/hot variables."""
    if not USE_MONOTONE:
        return None
    plus = {"hdd", "cdd", "hdd_x_slot", "hdd_x_weekend", "hdd_lag24", "cdd_lag24"}
    vec = [(1 if f in plus else 0) for f in feats]
    return "(" + ",".join(str(v) for v in vec) + ")"

def get_xgb(feats):
    kw = dict(n_estimators=600, learning_rate=0.03, max_depth=5, min_child_weight=3,
              subsample=0.9, colsample_bytree=0.9, reg_alpha=0.3, reg_lambda=2.0,
              objective=XGB_OBJECTIVE, tree_method="hist", random_state=SEED, n_jobs=-1)
    mono = build_monotone(feats)
    if mono is not None:
        kw["monotone_constraints"] = mono
    return XGBRegressor(**kw)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]
    if len(y_true) == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "R2": np.nan, "BIAS": np.nan, "N": 0}
    return {"RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAE":  float(mean_absolute_error(y_true, y_pred)),
            "R2":   float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
            "BIAS": float(np.mean(y_pred - y_true)), "N": int(len(y_true))}

### 5.2 Training the baseline

Key steps:
1. We compute the **residual target** with a profile **guaranteed NaN-free**.
2. **Anti-NaN fix**: we exclude from `fit` any row whose label or energy is non-finite (any grid gaps).
3. We apply **local debiasing** then **clipping**.
4. We rebuild the final prediction = `ref_profile + predicted residual`.

In [ ]:
def train_baseline(d, feats, train_mask, baseline_mask):
    d = d.copy()
    X = d[feats]
    prof = d["ref_profile"].fillna(float(np.nanmean(d["energy"])))
    y = (d["energy"] - prof) if RESIDUAL_TARGET else d["energy"].copy()

    # --- Anti-NaN fix: label and energy must be finite ---
    label_finite  = np.isfinite(y.values)
    energy_finite = np.isfinite(d["energy"].values)
    fit_mask = (train_mask & baseline_mask).values & label_finite & energy_finite
    fit_mask = pd.Series(fit_mask, index=d.index)
    if int(fit_mask.sum()) == 0:
        raise ValueError("No valid row for training.")

    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("model", get_xgb(feats))])
    pipe.fit(X.loc[fit_mask], y.loc[fit_mask])
    pred = pipe.predict(X)
    d["pred_baseline"] = (prof + pred) if RESIDUAL_TARGET else pred

    # Local debiasing (30 days before the test)
    if LOCAL_DEBIASING:
        deb = test_start - pd.Timedelta(days=DEBIAS_CALIB_DAYS)
        mcal = ((d["timestamp"] >= deb) & (d["timestamp"] < test_start)
                & baseline_mask & pd.Series(energy_finite, index=d.index))
        if mcal.sum() > STEPS_PER_DAY:
            bias = float((d.loc[mcal, "pred_baseline"] - d.loc[mcal, "energy"]).mean())
            if np.isfinite(bias):
                d["pred_baseline"] -= bias

    # Clipping
    if CLIP_PREDICTIONS:
        lo, hi = d["energy"].quantile([CLIP_Q_LOW, CLIP_Q_HIGH])
        d["pred_baseline"] = d["pred_baseline"].clip(lo, hi)
    return pipe, d

### 5.3 Full per-segment pipeline

`segment_pipeline(...)` chains all steps for one segment, then evaluates the baseline over the test window — first on **all days**, then on **clean days** (non-event days), where the baseline should closely match the actual consumption.

In [ ]:
def segment_pipeline(name, d_raw, weather):
    print(f"\n===== SEGMENT: {name} =====")
    d = add_event_features(d_raw)
    # Maintenance flags
    if DETECT_MAINTENANCE:
        fl = detect_maintenance(d)
        d = d.merge(fl[["date", "flag_maintenance"]], on="date", how="left")
        d["flag_maintenance"] = d["flag_maintenance"].fillna(0).astype(int)
    else:
        d["flag_maintenance"] = 0
    d = add_features(d, weather)

    test_mask  = d["timestamp"].between(test_start, test_end)
    train_mask = ~test_mask
    base_mask  = build_baseline_mask(d)
    d = add_reference_profile(d, train_mask & base_mask)

    pipe, d = train_baseline(d, FEATURES, train_mask, base_mask)

    d_eval = d.loc[test_mask].copy().reset_index(drop=True)
    met_all = compute_metrics(d_eval["energy"], d_eval["pred_baseline"])
    clean = (d_eval["is_event_day"] == 0) & (d_eval["flag_maintenance"] == 0)
    met_clean = compute_metrics(d_eval.loc[clean, "energy"], d_eval.loc[clean, "pred_baseline"])
    print("  Metrics (all days)  :", {k: round(v,4) if isinstance(v,float) else v for k,v in met_all.items()})
    print("  Metrics (clean days):", {k: round(v,4) if isinstance(v,float) else v for k,v in met_clean.items()})
    return {"segment": name, "model": pipe, "d_full": d, "d_eval": d_eval,
            "met_all": met_all, "met_clean": met_clean}

results = {}
for name, d_raw in segment_data.items():
    results[name] = segment_pipeline(name, d_raw, weather)

## 6. Performance summary

We gather the metrics of the 4 segments. The **R²** column on clean days indicates the baseline quality: the closer it is to 1, the better the baseline reproduces normal consumption.

In [ ]:
rows = []
for name, r in results.items():
    for scope, met in [("all_days", r["met_all"]), ("clean_days", r["met_clean"])]:
        rows.append({"segment": name, "scope": scope, **met})
summary = pd.DataFrame(rows)
print(summary.round(4).to_string(index=False))

## 7. Visualization: Actual vs Baseline

For each segment, we plot the actual and the baseline over the test window. The **colored bands** mark event days: the baseline should **deviate from the actual** on those days (it predicts the "no-event" scenario), and **match** it on the other days.

In [ ]:
COLORS = {"Residential": "royalblue", "Business": "darkorange",
          "Controlled": "crimson", "Behavioral": "seagreen"}

def plot_baseline(r):
    d = r["d_eval"]
    fig, ax = plt.subplots(figsize=(16, 4.5))
    ax.plot(d["timestamp"], d["energy"], color="black", lw=1.1, label="Actual")
    ax.plot(d["timestamp"], d["pred_baseline"], color=COLORS[r["segment"]],
            lw=1.2, ls="--", label="XGBoost baseline")
    for j in d.loc[d["is_event_day"] == 1, "timestamp"].dt.normalize().unique():
        ax.axvspan(j, j + pd.Timedelta(days=1), color=COLORS[r["segment"]], alpha=0.10, lw=0)
    ax.axvline(DATE_TEST_TS, color="green", ls=":", lw=1.4, label="DATE_TEST")
    ax.set_title(f"{r['segment']} — Actual vs Baseline (15 min)")
    ax.set_ylabel("Energy"); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    ax.legend(loc="best"); plt.tight_layout(); plt.show()

for r in results.values():
    plot_baseline(r)

## 8. Load-shifting balance

We quantify, for each event day, the **curtailed energy** (deficit on D), the **pre-charge** (D-1) and the **rebound** (D+1), then the **net avoided energy**. This is the final business deliverable: how much energy the event actually saved once deferred load is taken into account.

In [ ]:
def load_shift_balance(r):
    d = r["d_full"]
    days = sorted(d.loc[(d["is_event_day"] == 1) &
                        d["timestamp"].between(test_start, test_end), "date"].unique())
    rows = []
    for j in days:
        j = pd.Timestamp(j)
        rec = {"segment": r["segment"], "event_date": j.date().isoformat()}
        for label, day in [("D-1", j-pd.Timedelta(days=1)), ("D", j), ("D+1", j+pd.Timedelta(days=1))]:
            z = d[d["date"] == day]
            gap = float((z["pred_baseline"] - z["energy"]).sum()) if len(z) else np.nan
            rec[f"gap_{label}"] = gap
        deficit   = rec.get("gap_D", 0.0)
        precharge = max(0.0, -rec.get("gap_D-1", 0.0))
        rebound   = max(0.0, -rec.get("gap_D+1", 0.0))
        rec["net_avoided_energy"] = deficit - precharge - rebound
        rows.append(rec)
    return pd.DataFrame(rows)

balances = pd.concat([load_shift_balance(r) for r in results.values()], ignore_index=True)
print(balances.round(3).to_string(index=False))

# Part 2 — TFT Baseline (Temporal Fusion Transformer)

## 9. Why a deep learning model?

XGBoost treats each slot **independently**. The **TFT**, on the other hand, is designed for **time series**: it looks at a **past window** (encoder) to predict the **future** (decoder), and learns fine temporal dependencies.

### Leakage-free TFT baseline principle
- **Encoder** = 7 **real** days preceding the day to predict (context).
- **Decoder** = 96 slots of the target day, fed **only** by the **calendar + weather** (variables known in advance).
- ❗ The decoder **never** sees the event → **no leakage**: the prediction is truly a "no-event" counterfactual.

> ⚙️ **Execution**: this part requires `torch`, `lightning` and `pytorch-forecasting`. On Google Colab with a GPU, it installs and runs directly. The code below installs automatically and guards against a GPU-less environment.

In [ ]:
# Automatic install of the deep learning stack (Colab)
import importlib, subprocess, sys
def _ensure(pkg, pip_name=None):
    try:
        return importlib.import_module(pkg)
    except ModuleNotFoundError:
        print(f"Installing {pip_name or pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])
        return importlib.import_module(pkg)

TFT_AVAILABLE = True
try:
    torch = _ensure("torch")
    _ensure("lightning", "lightning")
    _ensure("pytorch_forecasting", "pytorch-forecasting")
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import QuantileLoss
    GPU = torch.cuda.is_available()
    print("torch", torch.__version__, "| GPU:", GPU)
except Exception as e:
    TFT_AVAILABLE = False
    print("TFT unavailable in this environment:", e)
    print("-> Part 2 will be skipped. Run this notebook on Colab (GPU) to enable it.")

## 10. TFT hyperparameters and data preparation

We set the window lengths (encoder 7 d = 672 steps, decoder 1 d = 96 steps) and the network hyperparameters. For a pedagogical notebook, we keep a **lightweight** model (few epochs): the goal is to understand the mechanism, not to reach maximum performance.

`prepare_tft_df(...)` puts the data on a contiguous grid, creates an integer `time_idx` and **interpolates** any NaNs — a strict requirement of `pytorch-forecasting`.

In [ ]:
ENC_DAYS, DEC_DAYS = 7, 1
ENC_LEN = ENC_DAYS * STEPS_PER_DAY      # 672
DEC_LEN = DEC_DAYS * STEPS_PER_DAY      # 96
MAX_EPOCHS   = 8        # pedagogical (increase to 30+ for better results)
BATCH_SIZE   = 128
HIDDEN_SIZE  = 16
ATTN_HEADS   = 2
DROPOUT      = 0.15

KNOWN_REALS = ["slot", "dow", "month", "is_weekend",
               "slot_sin", "slot_cos", "dow_sin", "dow_cos",
               "hdd", "cdd", "humidex", "wind", "temperature"]

def prepare_tft_df(d_full):
    d = d_full.copy().sort_values("timestamp").reset_index(drop=True)
    grid = pd.date_range(d["timestamp"].min(), d["timestamp"].max(), freq=FREQ)
    d = d.set_index("timestamp").reindex(grid).rename_axis("timestamp").reset_index()
    d["energy"] = d["energy"].interpolate(limit_direction="both")
    for c in KNOWN_REALS:
        if c in d.columns:
            d[c] = d[c].interpolate(limit_direction="both").ffill().bfill()
    d["time_idx"] = np.arange(len(d), dtype=np.int64)
    d["series"] = "s0"
    d["date"] = d["timestamp"].dt.normalize()
    d["is_event_day"] = d["is_event_day"].fillna(0).astype(int)
    d["event"] = d["event"].fillna(0).astype(int)
    known = [c for c in KNOWN_REALS if c in d.columns]
    return d, known

if TFT_AVAILABLE:
    print(f"Encoder={ENC_LEN} steps ({ENC_DAYS} d) | Decoder={DEC_LEN} steps | epochs={MAX_EPOCHS}")

## 11. TFT training and rolling baseline

- `train_tft(...)` trains the model on the history **before** `test_start` (with a small validation split), never seeing the test window.
- `rolling_tft_baseline(...)` advances **day by day**: for each day D in the test window, it takes 7 real days as encoder and predicts the 96 slots of D from the calendar + weather only.

> The whole block is guarded by `if TFT_AVAILABLE` so the notebook never crashes if the deep learning stack is absent.

In [ ]:
def train_tft(df, known):
    cutoff = int(df.loc[df["timestamp"] >= test_start, "time_idx"].min())
    df_hist = df[df["time_idx"] < cutoff].copy()
    val_start = df_hist["time_idx"].max() - int(0.10 * len(df_hist))
    ds_train = TimeSeriesDataSet(
        df_hist[df_hist["time_idx"] <= val_start],
        time_idx="time_idx", target="energy", group_ids=["series"],
        max_encoder_length=ENC_LEN, max_prediction_length=DEC_LEN,
        time_varying_known_reals=known,
        time_varying_unknown_reals=["energy"],
        target_normalizer=GroupNormalizer(groups=["series"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=True,
    )
    ds_val = TimeSeriesDataSet.from_dataset(ds_train, df_hist, predict=False,
                                            stop_randomization=True,
                                            min_prediction_idx=val_start + 1)
    dl_train = ds_train.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=0)
    dl_val   = ds_val.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
    tft = TemporalFusionTransformer.from_dataset(
        ds_train, learning_rate=0.01, hidden_size=HIDDEN_SIZE,
        attention_head_size=ATTN_HEADS, dropout=DROPOUT,
        loss=QuantileLoss(), log_interval=-1, optimizer="adam")
    trainer = pl.Trainer(max_epochs=MAX_EPOCHS,
                         accelerator=("gpu" if GPU else "cpu"), devices=1,
                         gradient_clip_val=0.1, enable_model_summary=False,
                         callbacks=[EarlyStopping(monitor="val_loss", patience=3, mode="min")],
                         logger=False, enable_checkpointing=False)
    trainer.fit(tft, train_dataloaders=dl_train, val_dataloaders=dl_val)
    return tft, ds_train

def rolling_tft_baseline(df, model, ds_train):
    days = pd.date_range(test_start.normalize(), test_end.normalize(), freq="D")
    parts = []
    for D in days:
        end_D = D + pd.Timedelta(days=1) - pd.Timedelta(minutes=15)
        sl = df[(df["timestamp"] >= D - pd.Timedelta(days=ENC_DAYS)) &
                (df["timestamp"] <= end_D)].copy()
        if len(sl) < ENC_LEN + DEC_LEN:
            continue
        sl = sl.iloc[-(ENC_LEN + DEC_LEN):].copy()
        ds = TimeSeriesDataSet.from_dataset(ds_train, sl, predict=True, stop_randomization=True)
        dl = ds.to_dataloader(train=False, batch_size=1, num_workers=0)
        yhat = np.asarray(model.predict(dl, mode="prediction")).reshape(-1)[:DEC_LEN]
        day = sl.iloc[-DEC_LEN:][["timestamp"]].copy()
        day["pred_tft"] = yhat
        parts.append(day)
    if not parts:
        return pd.DataFrame(columns=["timestamp", "pred_tft"])
    return pd.concat(parts, ignore_index=True)

results_tft = {}
if TFT_AVAILABLE:
    for name, r in results.items():
        print(f"\n===== TFT: {name} =====")
        df_tft, known = prepare_tft_df(r["d_full"])
        model, ds_train = train_tft(df_tft, known)
        base_tft = rolling_tft_baseline(df_tft, model, ds_train)
        d_eval = r["d_eval"][["timestamp", "energy", "pred_baseline",
                              "event", "is_event_day"]].copy()
        d_eval = d_eval.merge(base_tft, on="timestamp", how="left")
        results_tft[name] = {"segment": name, "d_eval": d_eval}
        print(f"  TFT-predicted points: {d_eval['pred_tft'].notna().sum()} / {len(d_eval)}")
else:
    print("Part 2 (TFT) skipped: environment without deep learning.")

## 12. XGBoost vs TFT comparison

We compare both baselines **only on clean days** (non-event days), the right ground for evaluating a counterfactual. The winner is determined by **R²**.

In [ ]:
if TFT_AVAILABLE and results_tft:
    rows = []
    for name, r in results_tft.items():
        clean = r["d_eval"][r["d_eval"]["is_event_day"].fillna(0).astype(int) == 0]
        for mn, col in [("XGBoost", "pred_baseline"), ("TFT", "pred_tft")]:
            sub = clean.dropna(subset=[col])
            rows.append({"segment": name, "model": mn, **compute_metrics(sub["energy"], sub[col])})
    comparison = pd.DataFrame(rows)
    print(comparison.round(4).to_string(index=False))

    winner = []
    for name in results_tft:
        rx = comparison.query("segment==@name and model=='XGBoost'")["R2"].values[0]
        rt = comparison.query("segment==@name and model=='TFT'")["R2"].values[0]
        winner.append({"segment": name, "R2_XGBoost": round(rx,4), "R2_TFT": round(rt,4),
                       "winner": "TFT" if rt > rx else "XGBoost", "R2_gain": round(rt-rx,4)})
    print("\n", pd.DataFrame(winner).to_string(index=False))
else:
    print("Comparison unavailable (TFT not executed in this environment).")
    print("XGBoost metrics remain available in the 'summary' table (Section 6).")

## 13. Conclusion

### What this notebook demonstrated
1. **Simulation** of realistic 15-min consumption data, with ground truth (events, maintenance, weather).
2. **Automatic detection** of atypical days.
3. **Feature engineering**: calendar, cyclical encoding, lagged weather, anti-NaN reference profiles.
4. **Robust XGBoost baseline**: residual target, physical monotonicity, debiasing, clipping.
5. **Leakage-free TFT baseline**, in rolling day-by-day prediction.
6. **Honest comparison** on clean days and **load-shifting balance**.

### Key design points
- **Anti-leakage**: the TFT decoder only sees calendar + weather, never the event.
- **NaN robustness**: reference profile guaranteed finite, non-finite rows excluded from `fit`.
- **Physical coherence**: monotonic constraints on HDD/CDD.
- **Honest evaluation**: comparison only on clean days.

### Future work
- Expose the TFT's **prediction intervals** (via `QuantileLoss`).
- **Temporal cross-validation** over several event dates.
- Increase `MAX_EPOCHS` and TFT depth for better performance.
- Track runs with **MLflow**.

> 🔒 This notebook is fully **anonymized** and runs on **simulated data**. It serves as a pedagogical foundation and a portfolio deliverable, transferable to real data by simply replacing the simulation section with a data load (SQL / data warehouse).